# NuGet Audit SCA — Raw Tool Output Extraction

Clone the C# training repository, execute `dotnet package list --include-transitive --vulnerable --format json`, preserve the raw JSON output, and map findings to Dependency Risk (SCA) metrics.

## Cell 1 – Project Information

In [ ]:
# Display repository metadata and execution context.
from datetime import datetime, timezone
from pathlib import Path

REPO_URL = 'https://github.com/visvantha-testable/csharp-testing-nuget-audit'
PROGRAMMING_LANGUAGE = 'C#'
TOOL_NAME = 'dotnet package list --include-transitive --vulnerable --format json'
ANALYSIS_TYPE = 'Software Composition Analysis (SCA)'
EXECUTION_DATE = datetime.now(timezone.utc).strftime('%Y-%m-%d %H:%M:%S UTC')
WORKING_DIRECTORY = str(Path('.').resolve())

PROJECT_INFO = {
    'Repository URL': REPO_URL,
    'Programming Language': PROGRAMMING_LANGUAGE,
    'Tool Name': TOOL_NAME,
    'Analysis Type': ANALYSIS_TYPE,
    'Execution Date': EXECUTION_DATE,
    'Working Directory': WORKING_DIRECTORY,
}
for key, value in PROJECT_INFO.items():
    print(f'{key}: {value}')

## Cell 2 – Clone Repository

In [ ]:
# Install dependencies, import helpers, and clone the repository.
import json
import subprocess
import sys
from pathlib import Path

from IPython.display import display

subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', '-r', 'requirements.txt'])

METRIC_ROOT = Path('.').resolve()
if not (METRIC_ROOT / 'tool' / '_nuget_audit_sca_utils.py').exists():
    METRIC_ROOT = Path('..').resolve()
sys.path.insert(0, str(METRIC_ROOT / 'tool'))

from _nuget_audit_sca_utils import (
    ANALYSIS_TYPE,
    AUDIT_PROJECT_RELATIVE,
    NO_EVIDENCE_MESSAGE,
    PROGRAMMING_LANGUAGE,
    REPO_URL,
    TOOL_NAME,
    NotebookLogger,
    build_evidence_table,
    build_final_summary,
    build_metric_mapping,
    clone_repository,
    collect_prerequisite_versions,
    discover_solution,
    dotnet_env,
    download_dotnet_sdk,
    ensure_output_dirs,
    export_results,
    list_repository_structure,
    parse_audit_json,
    preserve_raw_audit_output,
    read_text,
    resolve_metric_root,
    run_dotnet_build,
    run_dotnet_restore,
    run_nuget_audit,
)

METRIC_ROOT = resolve_metric_root(METRIC_ROOT)
DIRS = ensure_output_dirs(METRIC_ROOT)
OUTPUT_DIR = DIRS['output']
WORKSPACE_DIR = DIRS['workspace']
LOGGER = NotebookLogger(OUTPUT_DIR / 'error_log.txt')

REPO_PATH, CLONE_STATUS = clone_repository(REPO_URL, WORKSPACE_DIR, reuse=True)
print(CLONE_STATUS)
display(list_repository_structure(REPO_PATH))

## Cell 3 – Install Prerequisites

In [ ]:
# Verify .NET SDK, Git, Python, and notebook dependencies.
DOTNET_ROOT = download_dotnet_sdk(DIRS['runtimes'], tmp_dir=DIRS['tmp'])
DOTNET_ENV = dotnet_env(DOTNET_ROOT, tmp_dir=DIRS['tmp'])
PREREQ_DF = collect_prerequisite_versions(DOTNET_ROOT, DOTNET_ENV)
display(PREREQ_DF)
PREREQ_DF.to_csv(OUTPUT_DIR / 'prerequisite_versions.csv', index=False)

## Cell 4 – Restore NuGet Packages

In [ ]:
# Restore NuGet packages for the cloned solution.
SOLUTION_PATH = discover_solution(REPO_PATH)
RESTORE_RESULT = run_dotnet_restore(REPO_PATH, SOLUTION_PATH, DOTNET_ROOT, DOTNET_ENV)
print('--- restore logs ---')
print(RESTORE_RESULT['raw'])
print(f"Number of restored packages: {RESTORE_RESULT['restored_packages']}")
print(f"Restore status: {RESTORE_RESULT['restore_status']}")
(OUTPUT_DIR / 'dotnet_restore.log').write_text(RESTORE_RESULT['raw'], encoding='utf-8')
if not RESTORE_RESULT['success']:
    raise RuntimeError('dotnet restore failed.')

## Cell 5 – Build the Project

In [ ]:
# Build the cloned solution without modifying repository source code.
BUILD_RESULT = run_dotnet_build(REPO_PATH, SOLUTION_PATH, DOTNET_ROOT, DOTNET_ENV)
print('--- build logs ---')
print(BUILD_RESULT['raw'])
print(f"Number of projects: {BUILD_RESULT['project_count']}")
print(f"Build status: {BUILD_RESULT['build_status']}")
(OUTPUT_DIR / 'dotnet_build.log').write_text(BUILD_RESULT['raw'], encoding='utf-8')
if not BUILD_RESULT['success']:
    raise RuntimeError('dotnet build failed.')

## Cell 6 – Execute NuGet Audit

In [ ]:
# Run the native NuGet Audit command and capture raw JSON output.
AUDIT_RESULT = run_nuget_audit(REPO_PATH, AUDIT_PROJECT_RELATIVE, DOTNET_ROOT, DOTNET_ENV)
print('--- console output ---')
print(AUDIT_RESULT['raw'])
print('--- raw JSON output ---')
print(AUDIT_RESULT['stdout'])
print(f"Audit status: {AUDIT_RESULT.get('audit_status')}")
if not AUDIT_RESULT['success'] or not AUDIT_RESULT['stdout'].strip():
    raise RuntimeError('NuGet audit command failed.')

## Cell 7 – Preserve Raw Tool Output

In [ ]:
# Save raw JSON, console output, and audit logs without modification.
RAW_PATHS = preserve_raw_audit_output(AUDIT_RESULT, OUTPUT_DIR)
print('===== Raw JSON (verbatim) =====')
print(read_text(RAW_PATHS['raw_json']))
print('===== Raw console output (verbatim) =====')
print(read_text(RAW_PATHS['raw_console']))
print('===== Audit execution log (verbatim) =====')
print(read_text(RAW_PATHS['audit_log']))

## Cell 8 – Parse JSON Output

In [ ]:
# Parse every dependency and vulnerability record from the audit JSON.
AUDIT_PAYLOAD = json.loads(AUDIT_RESULT['stdout'])
FINDINGS_DF = parse_audit_json(AUDIT_RESULT['stdout'], AUDIT_PAYLOAD)
display(FINDINGS_DF)

## Cell 9 – Metric Mapping

In [ ]:
# Map NuGet Audit JSON fields to Dependency Risk (SCA) metrics.
METRIC_MAPPINGS = build_metric_mapping(FINDINGS_DF, AUDIT_PAYLOAD)
for mapping in METRIC_MAPPINGS:
    print(f"\nMetric: {mapping['metric']}")
    print(f"Classification: {mapping['classification']}")
    print(f"Technique: {mapping['technique']}")
    if mapping['has_evidence']:
        print('Supporting dependency entries:')
        display(mapping['supporting_rows'])
        print(f"Rationale: {mapping['rationale']}")
    else:
        print(NO_EVIDENCE_MESSAGE)
        print(f"Rationale: {mapping['rationale']}")

## Cell 10 – Evidence Table

In [ ]:
# Build the metric evidence table directly from parsed audit JSON rows.
EVIDENCE_DF = build_evidence_table(FINDINGS_DF, METRIC_MAPPINGS)
display(EVIDENCE_DF)

## Cell 11 – Export Results

In [ ]:
# Export raw and parsed outputs to the output/ directory.
SUMMARY = build_final_summary(
    REPO_PATH,
    FINDINGS_DF,
    METRIC_MAPPINGS,
    AUDIT_PAYLOAD,
    RESTORE_RESULT,
    BUILD_RESULT,
)
EXPORTED = export_results(
    OUTPUT_DIR,
    RAW_PATHS,
    FINDINGS_DF,
    EVIDENCE_DF,
    METRIC_MAPPINGS,
    SUMMARY,
)
for name, path in EXPORTED.items():
    print(f'{name}: {path}')

## Cell 12 – Final Summary

In [ ]:
# Display the final execution summary derived from NuGet Audit output.
print(f"Repository Name: {SUMMARY['repository_name']}")
print(f"Programming Language: {SUMMARY['programming_language']}")
print(f"Tool Used: {SUMMARY['tool_used']}")
print(f"Total Projects Analysed: {SUMMARY['total_projects_analysed']}")
print(f"Total Dependencies: {SUMMARY['total_dependencies']}")
print(f"Direct Dependencies: {SUMMARY['direct_dependencies']}")
print(f"Transitive Dependencies: {SUMMARY['transitive_dependencies']}")
print(f"Vulnerable Packages: {SUMMARY['vulnerable_packages']}")
print(f"Critical Vulnerabilities: {SUMMARY['critical_vulnerabilities']}")
print(f"High Vulnerabilities: {SUMMARY['high_vulnerabilities']}")
print(f"Medium Vulnerabilities: {SUMMARY['medium_vulnerabilities']}")
print(f"Low Vulnerabilities: {SUMMARY['low_vulnerabilities']}")
print(f"Metrics Evaluated: {SUMMARY['metrics_evaluated']}")
print(f"Metrics with Supporting Evidence: {SUMMARY['metrics_with_supporting_evidence']}")
print(f"Metrics without Supporting Evidence: {SUMMARY['metrics_without_supporting_evidence']}")
LOGGER.write_errors()